In [1]:
import sys, os
from pathlib import Path

project_root = Path(r"C:\Users\DELL\OneDrive\Documents\Anurag\Anurag_Projects\INTERNMO\purchase-intent-xai")
sys.path.append(str(project_root))
os.chdir(project_root)

print("Working directory:", os.getcwd())
print("src exists at that path:", (project_root / "src").exists())

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from src.config.settings import CONFIG
from src.data.loader import load_sessions_data
from src.features.engineer import engineer_features

pd.set_option("display.max_columns", None)

print("All imports successful.")

Working directory: C:\Users\DELL\OneDrive\Documents\Anurag\Anurag_Projects\INTERNMO\purchase-intent-xai
src exists at that path: True
All imports successful.


In [2]:
df = load_sessions_data()
df = engineer_features(df)
df.shape

(12000, 23)

In [3]:
target = CONFIG["target_column"]
categorical_columns = CONFIG["categorical_columns"]
feature_columns = [c for c in df.columns if c not in categorical_columns + [target]]

# Quick one-hot for this diagnostic only — your real preprocessing pipeline handles this properly in Week 2 modeling
X = pd.get_dummies(df[feature_columns + categorical_columns], columns=categorical_columns)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["split"]["test_size"],
    stratify=y, random_state=CONFIG["random_seed"]
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (9600, 80)
X_test shape: (2400, 80)


Data split into 9,600 train / 2,400 test rows (80/20) with stratification on Converted, ensuring class balance is preserved across both sets — one-hot encoding produced 80 feature columns, a diagnostic split only (real preprocessing handled in Week 2 modeling).

In [4]:
rf = RandomForestClassifier(n_estimators=200, random_state=CONFIG["random_seed"])
rf.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [5]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(15)

PageValues                 0.341589
ProductRelated             0.086583
TotalPages                 0.069474
ExitRates                  0.062047
BounceRates                0.046177
TotalDuration              0.043567
ProductRelated_Duration    0.041182
AvgTimePerProductPage      0.034100
ProductPageRatio           0.029853
Administrative_Duration    0.029309
Informational_Duration     0.017655
Administrative             0.014880
Informational              0.007424
SpecialDay                 0.005494
OperatingSystems_1         0.005429
dtype: float64

PageValues dominates feature importance (0.34, ~4x the next feature) — consistent with the earlier leakage concern, so its outsized influence here likely inflates model performance rather than reflecting genuine predictive skill.

Next-strongest features (ProductRelated, TotalPages, ExitRates, BounceRates) are more modest and behavior-based, making them safer candidates to lean on if PageValues is dropped or down-weighted.

In [6]:
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=CONFIG["random_seed"], n_jobs=-1
)
perm_importances = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
perm_importances.head(15)

PageValues                 0.104167
ProductRelated             0.007458
TotalPages                 0.004375
BounceRates                0.002792
ExitRates                  0.002250
Administrative_Duration    0.001042
ProductPageRatio           0.000875
Informational_Duration     0.000750
Browser_2                  0.000750
ProductRelated_Duration    0.000708
Administrative             0.000667
TrafficType_2              0.000500
OperatingSystems_1         0.000458
Browser_13                 0.000417
Informational              0.000417
dtype: float64

Permutation importance confirms PageValues as by far the strongest signal (0.104, ~14x the next feature) — even more pronounced than the built-in feature importances, reinforcing that model performance is heavily reliant on this leakage-risk feature.

All other features contribute marginally (<0.008) — the model is essentially learning one dominant signal, so results should be re-validated after removing/adjusting PageValues.

In [7]:
print("Model-based rank:", importances.rank(ascending=False)["IsReturningVisitor"])
print("Permutation rank:", perm_importances.rank(ascending=False)["IsReturningVisitor"])
print("Model-based value:", importances["IsReturningVisitor"])
print("Permutation value:", perm_importances["IsReturningVisitor"])

Model-based rank: 48.0
Permutation rank: 39.0
Model-based value: 0.002296581805089163
Permutation value: 4.166666666665098e-05


IsReturningVisitor ranks 48th (model-based) and 39th (permutation) out of 80 features — both values are small (0.0023 and 0.00004), confirming it adds little on top of what's already captured. This makes sense: VisitorType's one-hot encoding (including VisitorType_Returning_Visitor) already carries this exact information

In [8]:
comparison = pd.DataFrame({
    "model_based_rank": importances.rank(ascending=False),
    "permutation_rank": perm_importances.rank(ascending=False),
}).sort_values("model_based_rank")

comparison.head(20)

,model_based_rank,permutation_rank
PageValues,1.0,1.0
ProductRelated,2.0,2.0
TotalPages,3.0,3.0
ExitRates,4.0,5.0
BounceRates,5.0,4.0
TotalDuration,6.0,80.0
ProductRelated_Duration,7.0,10.0
AvgTimePerProductPage,8.0,74.0
ProductPageRatio,9.0,7.0
Administrative_Duration,10.0,6.0


Top 3 features (PageValues, ProductRelated, TotalPages) agree strongly across both methods, but several features diverge sharply — flagging engineered/derived features that model-based importance overrates:

TotalDuration: rank 6 (model) vs 80 (permutation) — biggest gap, likely correlated/redundant with other duration features
AvgTimePerProductPage: rank 8 vs 74 — same issue
Browser_1: rank 16 vs 68
Region_1: rank 18 vs 43
SpecialDay: rank 14 vs 28

In [ ]:
Final feature set for Week 2 modeling: Keeping the original 18 columns plus TotalPages and ProductPageRatio, which perform consistently well across both importance lenses. Dropping TotalDuration and AvgTimePerProductPage (severely overrated by model-based importance vs. permutation — ranks 80 and 74) and IsReturningVisitor (redundant with existing VisitorType encoding, ranks 48/39 out of 80 with negligible importance values).